# Data Exploration Notebook

Use this notebook to explore the raw data before you start building your pipeline or models.

**Prerequisites:** Materialize the `ingestion` asset group in Dagster first (`pixi run start-dev`, then click Materialize All on the ingestion group). This writes the raw tables to DuckDB.

Or run the cells below to fetch directly from the API without Dagster.

In [ ]:
import json
import os

import duckdb
import pandas as pd
import requests

BASE_URL = "https://fakestoreapi.com"
DB_PATH = os.getenv("ANALYTICS_DB_PATH", "../analytics_database_dev.duckdb")

## Option A: Read from DuckDB (after Dagster materialization)

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)

try:
    tables = con.execute("SHOW TABLES").fetchdf()
    print("Tables in DuckDB:")
    display(tables)
except Exception as e:
    print(f"Could not connect to DuckDB: {e}")
    print("Run the ingestion assets in Dagster first, or use Option B below.")

In [ ]:
# Read raw tables
try:
    products_df = con.execute("SELECT * FROM raw_products").fetchdf()
    users_df    = con.execute("SELECT * FROM raw_users").fetchdf()
    carts_df    = con.execute("SELECT * FROM raw_carts").fetchdf()
    print(f"Products: {len(products_df)} rows")
    print(f"Users:    {len(users_df)} rows")
    print(f"Carts:    {len(carts_df)} rows")
except Exception:
    print("Tables not found. Use Option B to fetch from the API directly.")

## Option B: Fetch directly from the FakeStore API

In [ ]:
def fetch(endpoint: str) -> list[dict]:
    resp = requests.get(f"{BASE_URL}/{endpoint}", timeout=30)
    resp.raise_for_status()
    return resp.json()

products_raw = fetch("products")
users_raw    = fetch("users")
carts_raw    = fetch("carts")

print(f"Products: {len(products_raw)}")
print(f"Users:    {len(users_raw)}")
print(f"Carts:    {len(carts_raw)}")

## Products

In [ ]:
products_df = pd.json_normalize(products_raw)
products_df.head()

In [ ]:
print("Schema:")
print(products_df.dtypes)
print("\nCategories:")
print(products_df["category"].value_counts())

## Users

In [ ]:
users_df = pd.json_normalize(users_raw)
users_df.head()

In [ ]:
print("Schema:")
print(users_df.dtypes)

## Carts (Orders)

Note the nested `products` field — this is a list of `{productId, quantity}` dicts.
This is the structure you will need to handle in your transformation layer.

In [ ]:
carts_df = pd.DataFrame([
    {
        "id": c["id"],
        "user_id": c["userId"],
        "date": pd.to_datetime(c["date"]),
        "products": json.dumps(c["products"]),  # stored as JSON string in DuckDB
    }
    for c in carts_raw
])
carts_df.head()

In [ ]:
# Peek at the nested products structure for the first cart
first_cart_products = json.loads(carts_df.iloc[0]["products"])
print("First cart products:")
print(json.dumps(first_cart_products, indent=2))

In [ ]:
print(f"Date range: {carts_df['date'].min().date()} → {carts_df['date'].max().date()}")
print(f"Unique users with orders: {carts_df['user_id'].nunique()}")
print(f"Total carts: {len(carts_df)}")

## Your exploration

Add your own exploration cells below. Some questions to get you started:
- What is the price distribution across categories?
- Are there any users without orders?
- What is the maximum number of line items in a single cart?
- Are there any data quality issues you notice?

In [ ]:
# Your exploration here
